In [10]:
from pathlib import Path

# Split settings (keep fixed for reproducibility)
TEST_SIZE = 0.20
RANDOM_STATE = 42
TRY_STRATIFY = True

# Demo threshold for turning probability into 0 or 1 predictions
# Lower threshold catches more stroke cases but creates more false alarms
THRESHOLD = 0.30

# Data location (relative to repo root)
DATA_REL_PATH = Path("data/raw/stroke_data.csv")

# Output report location (relative to repo root)
REPORT_REL_PATH = Path("reports/baseline_metrics.md")

# Known leakage columns to drop (only dropped if they exist in the dataset)
LEAKAGE_COLS = [
    "General health condition",
    "depression",
    "Minutes sedentary activity",
]

# Columns that are coded categories even though they are numbers
# Keep the exact spelling from the dataset (note the trailing space in alcohol)
CODED_CATEGORICAL_COLS = [
    "gender",
    "age",
    "Race",
    "Marital status",
    "alcohol ",
    "smoke",
    "sleep disorder",
    "Health Insurance",
    "diabetes",
    "hypertension",
    "high cholesterol",
    "Coronary Heart Disease",
    "Body Mass Index",
]

# Thresholds we quickly scan for a precision/recall tradeoff check
THRESHOLD_SCAN = [0.05, 0.10, 0.20, 0.30]


In [11]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from pathlib import Path

In [12]:
def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / DATA_REL_PATH).exists():
            return p
    return here

ROOT = find_repo_root()
DATA_PATH = ROOT / DATA_REL_PATH

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find dataset at: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df.head()

,stroke,gender,age,Race,Marital status,alcohol,smoke,sleep disorder,Health Insurance,General health condition,...,energy,protein,Carbohydrate,Dietary fiber,Total fat,Total saturated fatty acids,Total monounsaturated fatty acids,Total polyunsaturated fatty acids,Potassium,Sodium
0,0,2,2,5,1,0,0,2,2,3,...,1598,62.78,192.19,10.0,65.64,25.112,24.090,8.543,2887,2969
1,0,2,2,1,1,0,0,1,2,3,...,1547,45.35,256.02,17.0,42.56,13.423,15.389,10.613,2058,2091
2,1,1,2,3,1,1,1,2,1,3,...,2466,81.56,254.49,13.0,103.32,43.295,36.727,15.366,3117,5233
3,0,2,3,3,1,1,1,2,1,4,...,1605,70.99,143.37,10.0,81.60,24.527,30.567,18.174,1766,3706
4,0,1,1,4,1,0,0,2,1,2,...,1818,74.75,229.45,14.2,67.49,26.030,24.837,10.533,1842,2461
